# 20 · Triage a support inbox

## The problem

Every support ticket needs three decisions before anyone can help: which **queue** owns it, how **urgent** it is, and
whether a manager must be pulled in **now** (a legal threat, a security breach, a big account about to churn). Keyword
rules misroute (*"charge"* appears in a fraud report too), and calling an LLM per ticket is slow and expensive at inbox
volume, and gives you prose you then have to parse.

## Why Jev

All three decisions are **typed**: pick one queue, place the ticket on an urgency scale, answer yes/no on escalation. Jev
answers exactly those question types with **calibrated probabilities** in about 100 ms, for a fraction of a cent, and all
three go in **one call** per ticket. Your code then sorts, colours and pages people based on plain thresholds.

And because the ticket leaves your system, customer identifiers are **redacted before** the call.

## What we'll build

```
ticket ──► redact emails/phones ──► Jev: 3 judgments in ONE call ──► sort by urgency
                                     • queue    (Choice)              ├─ escalate p > 0.6  ──► page a manager
                                     • urgency  (Score 0..3)          └─ urgency ≥ 2.5     ──► top of the queue
                                     • escalate (Noul)
```

## What you'll learn
- The three Jev question types side by side: `Choice` for the queue, `Score` for urgency, `Noul` for escalation.
- Why **urgency and escalation are separate** questions, and how they disagree in useful ways.
- How to **measure** a triage system: missed-urgent and over-escalation are different errors with different costs.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #14.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the Jev API key (`TYPESAFE_API_KEY`) from the `.env` file at the repo root,
- creates the `jev` client (one object, reused for every call),
- imports the question types. In this notebook we use **`Noul`**, `Score` and `Choice` (all three types). `NoulCriteria` is imported for the exercises,
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so the outputs are easy to read at a glance.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · Redact identifiers first

Before a ticket goes anywhere, we replace email addresses with `[EMAIL]` and phone numbers with `[PHONE]` using two
regexes. Jev doesn't need to know *who* wrote in to decide *what* the ticket is about, so we don't send it.

> **Workshop note:** the original email pattern also swallowed a full stop right after an address (`sam@shop.io.` became
> `[EMAIL]` and the sentence lost its period). The domain part now only matches dot-separated words, and the second test
> line checks that case.

These are simple patterns for the workshop. A production redactor would also cover names, addresses, account IDs and so on.

In [2]:
def redact(text):
    text = re.sub(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", "[EMAIL]", text)
    text = re.sub(r"\+?\d[\d\s().-]{7,}\d", "[PHONE]", text)
    return text


print(redact("Hi, I'm dana@bigco.com, call me at +1 (415) 555-0199"))
print(redact("Hi, this is sam@shop.io. Our checkout is down."))

Hi, I'm [EMAIL], call me at [PHONE]
Hi, this is [EMAIL]. Our checkout is down.


## Step 2 · Three questions, one request

| Key | Type | What it returns | Used for |
|---|---|---|---|
| `queue` | `Choice` | `.choice` (winning label), `.confidence`, `.probabilities` (one per label) | which team gets the ticket |
| `urgency` | `Score` | `.score`, a continuous position from 0 (Low) to 3 (Critical), plus `.confidence` | sort order and colour |
| `escalate` | `Noul` | `.noul`, P(yes) from 0 to 1 | page a manager or not |

**How a `Score` works:** `criteria` is an **ordered rubric**. Level 0 is the first line, level 3 the last, and Jev returns a
continuous position on it, so 2.3 means "High, leaning towards Critical".

**Urgency vs escalation.** They sound alike but answer different questions. *Urgency* is "how fast must someone act?".
*Escalate* is "does this need a **manager**?", and its instructions list exactly what counts: legal threats, safety,
security breaches, a large customer threatening to leave. A full outage is critical but goes to on-call engineers, not
necessarily to a manager.

This cell only **defines** the questions. Nothing is sent to Jev yet.

In [3]:
TRIAGE = {
    "queue": Choice(
        instructions="Which team should own this ticket?",
        criteria={
            "billing": "Charges, invoices, refunds, plan changes",
            "technical": "Bugs, errors, outages, integrations",
            "account": "Login, password, access, account settings",
            "sales": "Pricing questions, upgrades, new purchases",
            "trust_safety": "Abuse, fraud, security incidents, harassment",
        },
    ),
    "urgency": Score(
        instructions="How urgent is this ticket?",
        criteria=["Low: a question, no impact", "Normal: one user inconvenienced",
                  "High: work blocked or money at stake", "Critical: outage, security or legal exposure"],
    ),
    "escalate": Noul(
        instructions="Does this ticket need a manager now: a legal threat, a safety issue, a security breach, or a large customer threatening to leave?",
    ),
}
print("defined", len(TRIAGE), "questions:", list(TRIAGE))

defined 3 questions: ['queue', 'urgency', 'escalate']


## Step 3 · Look at one raw answer

One ticket, all three answers. Notice the email in the ticket: `triage()` (next step) redacts it, and here we call
`redact()` by hand so you see exactly what Jev receives.

In [4]:
ticket = "Hi, this is sam@shop.io. Your API has been timing out for 2 hours and our checkout is down."
print("sent to Jev:", redact(ticket))

r = jev.system_one(state=redact(ticket), questions=TRIAGE)
q = r.choices["queue"]
print(f"\nqueue   : {q.choice}  (confidence {q.confidence:.2f})")
for label, p in sorted(q.probabilities.items(), key=lambda kv: -kv[1]):
    print(f"            {label:<13} {bar(p, 10)} {p:.2f}")
print(f"urgency : {r.scores['urgency'].score:.2f} / 3")
print(f"escalate: {r.nouls['escalate'].noul:.2f}")

sent to Jev: Hi, this is [EMAIL]. Your API has been timing out for 2 hours and our checkout is down.



queue   : technical  (confidence 1.00)
            technical     ██████████ 1.00
            billing       ·········· 0.00
            account       ·········· 0.00
            sales         ·········· 0.00
            trust_safety  ·········· 0.00
urgency : 2.99 / 3
escalate: 0.17


**What to notice**
- The email is gone before the request is made; the ticket still makes perfect sense without it.
- Queue `technical` wins outright, and `.probabilities` shows the whole distribution, not just the winner.
- Urgency is near the top of the scale (checkout down = money at stake, outage), yet `escalate` stays **low**: it's an engineering emergency, not a legal, security or churn threat. Two questions, two different answers, both sensible.

## Step 4 · The triage function and the policy

`triage()` redacts, makes the one call, and returns plain numbers. The **policy** is two thresholds in code:

| Signal | Threshold | Action |
|---|---|---|
| `escalate` | `> ESCALATE_AT` (0.6) | 🚨 page a manager |
| `urgency` | `≥ URGENT_AT` (2.5, between High and Critical) | treat as urgent: top of the queue, shown red |
| `urgency` | `≥ 1.3` | shown amber |

This cell defines the function and thresholds only.

In [5]:
ESCALATE_AT = 0.6
URGENT_AT = 2.5


def triage(ticket, client):
    if "TRIAGE" not in globals():
        raise RuntimeError("TRIAGE is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")
    r = client.system_one(state=redact(ticket), questions=TRIAGE)
    q = r.choices["queue"]
    return {"queue": q.choice, "queue_conf": q.confidence,
            "urgency": r.scores["urgency"].score, "escalate": r.nouls["escalate"].noul}


print("triage() ready")

triage() ready


## Step 5 · Triage an inbox, sorted by urgency

Seven tickets, from a how-to question to a legal threat. We triage each one, then sort the inbox so the most urgent is on
top, and mark the ones that would page a manager.

In [6]:
INBOX = [
    "How do I add a teammate to my workspace?",
    "You charged my card twice this month. Please refund the duplicate.",
    "Our whole team gets a 500 error on login since 9am. 200 people blocked!",
    "Someone logged into my account from Russia and changed my email!!",
    "We're a 5,000-seat customer and if the SSO issue isn't fixed by Friday we're moving to a competitor.",
    "Do you offer discounts for nonprofits?",
    "If you don't delete my data within 7 days I will report you to the regulator and my lawyer.",
]
results = sorted(((triage(t, jev), t) for t in INBOX), key=lambda x: -x[0]["urgency"])
for d, t in results:
    esc = "  🚨 ESCALATE" if d["escalate"] > ESCALATE_AT else ""
    print(f"urgency {d['urgency']:.2f}  escalate {d['escalate']:.2f}  {d['queue']:<12} {t[:56]}{esc}")

urgency 3.00  escalate 0.92  trust_safety Someone logged into my account from Russia and changed m  🚨 ESCALATE
urgency 2.99  escalate 0.22  technical    Our whole team gets a 500 error on login since 9am. 200 
urgency 2.98  escalate 0.87  trust_safety If you don't delete my data within 7 days I will report   🚨 ESCALATE
urgency 2.31  escalate 0.94  technical    We're a 5,000-seat customer and if the SSO issue isn't f  🚨 ESCALATE
urgency 1.89  escalate 0.09  billing      You charged my card twice this month. Please refund the 
urgency 0.02  escalate 0.03  account      How do I add a teammate to my workspace?
urgency 0.02  escalate 0.05  sales        Do you offer discounts for nonprofits?


**What to notice**
- Top of the inbox: the 200-person login outage, the account takeover and the legal threat, all near **Critical** (close to 3).
- Exactly three tickets **escalate**: the account takeover (security breach), the regulator-and-lawyer threat (legal), and the 5,000-seat customer about to leave (churn).
- The **5,000-seat SSO ticket** is the interesting one: urgency only *High* (an SSO issue with a Friday deadline), but escalation is near-certain. A single "priority" number would have hidden that.
- The **login outage** is the mirror image: maximum urgency, but **no** escalation, because an outage isn't on the escalate list. It goes to on-call engineering.
- The how-to and the nonprofit-discount question sink to the bottom near 0, routed to `account` and `sales`.

## Step 6 · Measure it: missed-urgent vs over-escalation

A triage system makes two kinds of mistakes, and they cost different things:

- **missed urgent**: a ticket that needed fast action was ranked low. A customer waits; this is the expensive one.
- **missed escalation**: a manager should have been paged and wasn't.
- **over-escalation**: a manager was paged for nothing. Cheap once, but if it happens often, people stop answering the pager.

So we count them **separately**. Below, a human has labelled which inbox tickets are truly urgent and which truly need
escalation; we compare against the policy from Step 4. In real life you'd do this on a few hundred labelled tickets.

In [7]:
TRUE_URGENT = {INBOX[2], INBOX[3], INBOX[6]}            # outage, account takeover, legal threat
TRUE_ESCALATE = {INBOX[3], INBOX[4], INBOX[6]}          # takeover, 5,000-seat churn, legal threat

pred_urgent = {t for d, t in results if d["urgency"] >= URGENT_AT}
pred_escalate = {t for d, t in results if d["escalate"] > ESCALATE_AT}

print("missed urgent     :", len(TRUE_URGENT - pred_urgent), [t[:40] for t in TRUE_URGENT - pred_urgent])
print("missed escalation :", len(TRUE_ESCALATE - pred_escalate), [t[:40] for t in TRUE_ESCALATE - pred_escalate])
print("over-escalation   :", len(pred_escalate - TRUE_ESCALATE), [t[:40] for t in pred_escalate - TRUE_ESCALATE])

missed urgent     : 0 []
missed escalation : 0 []
over-escalation   : 0 []


**What to notice**
- On this small inbox all three error counts are **zero**: the policy agrees with the human labels.
- Seven tickets prove nothing statistically. The point is the *shape* of the check: once you have labels, every threshold change can be scored on each error type separately.
- If you lowered `ESCALATE_AT` to catch more churn risks, the **over-escalation** line is where you'd see the price.

## Step 7 · Give it a face

Everything from here on is **presentation only**: a small style kit, a form wired to `triage()` with `ipywidgets`, and an
HTML card per ticket (queue badge, urgency badge, escalate badge) using the same thresholds as above. The triage logic
is untouched.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

`ticket_html()` renders one ticket. To avoid paying for the calls twice, it can reuse the result we already have, so
here we render the sorted inbox from Step 5 without calling Jev again.

In [9]:
def urgency_color(u):
    return RED if u >= URGENT_AT else AMBER if u >= 1.3 else SAGE


def ticket_html(ticket, d=None):
    d = d or triage(ticket, jev)
    col = urgency_color(d["urgency"])
    esc = badge("ESCALATE", RED) if d["escalate"] > ESCALATE_AT else ""
    urg = badge(f"urgency {d['urgency']:.1f}/3", col)
    return (f'<div style="{CARD}border-left:5px solid {col};">{badge(d["queue"], SLATE)} '
            f'{urg} {esc}'
            f'<div style="margin-top:6px;">{redact(ticket)}</div></div>')


display(HTML(box("".join(ticket_html(t, d) for d, t in results))))

## Step 8 · Try it yourself

Edit the ticket and press **Triage** (needs a live kernel). Good ones to try:
- The default (`sam@shop.io`, API timing out, checkout down) → `technical`, urgency near **Critical**, **no** escalation, and the email shows as `[EMAIL]`.
- `Someone is sending threatening messages to my staff through your chat widget.` → `trust_safety`, **ESCALATE**.
- `Can I switch from monthly to annual billing?` → `billing`, urgency near 0, no escalation.
- `I'll cancel if you can't fix the CSV import, it's so frustrating.` → a churn threat from an ordinary user: escalation lands in the **middle**, below the 0.6 line. Is that the call you'd want? This is where you'd tune.

In [10]:
form_app([("ticket", "New ticket", "Hi, this is sam@shop.io. Your API has been timing out for 2 hours and our checkout is down.", 3)],
         lambda ticket: box(ticket_html(ticket)), "Triage")

Output()

## Recap and exercises

**Key ideas**
1. **Three question types, one call.** `Choice` picks the queue, `Score` places urgency on a rubric, `Noul` answers escalate yes/no.
2. **Separate signals catch different things.** An outage is urgent but not escalated; a big customer's churn threat is escalated but only High urgency.
3. **Redact before you send.** The model needs the problem, not the person.
4. **Measure each error type separately.** Missed-urgent, missed-escalation and over-escalation have different costs.

**Exercises**
- Add a queue `feedback` ("Feature requests and general product feedback") and send *"It would be great if exports supported XLSX"*.
- Add a `sentiment` `Score` (calm → furious) and show it on the card. Does it change your view of the CSV-import ticket?
- Lower `ESCALATE_AT` to 0.4 and re-run Steps 5 and 6. What happens to over-escalation, and does the CSV-import ticket now page a manager?
- Extend `redact()` to replace order IDs like `ORD-12345` with `[ORDER]`.